# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])


lines['qty'] = pd.to_numeric(lines['qty'])

# TODO: assert the row count and the quantity total

assert len(lines) == 4, f"Expected 4 rows, got {len(lines)}"
assert lines['qty'].sum() == 7, f"Expected total quantity of 7, got {lines['qty'].sum()}"

display(lines)

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-07


My code above takes the nested list of the orders and then flattens it by using the pd.json function. Then the meta order function copies the parent-level order detials. It does this for all of the orders so that it is clear which order and vendor the specific item is from. Then the assert statements I have are just to confirm that the table has the 4 line item rows necessary.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
#TODO
vendors_df = pd.DataFrame(vendors_json)


joined = pd.merge(lines, vendors_df, on='vendor_id', how='left', indicator=True)


unmatched = joined[joined['_merge'] == 'left_only']
unmatched_vendors = unmatched['vendor_id'].unique().tolist()
unmatched_qty = unmatched['qty'].sum()

print(f"Unmatched vendor ID(s): {unmatched_vendors}")
print(f"Units attached to unmatched vendor(s): {unmatched_qty}")
display(joined)

Unmatched vendor ID(s): ['V-07']
Units attached to unmatched vendor(s): 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-07,NaN,NaN,left_only


Here the first line takes my list of the vendor metadata and converts it into a structured Pandas DataFrame. Then pd.merge creates a left join so then it takes both the flattened order lines and the vendor details by using the vendor_id for the matching. Then my code filters the merged table so that it is can catch the V-07 is missing from my list but has 3 hot gods attached to it.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
#TODO
#I grouped by the vendor name, so that any of the unmatched vendors from the left-join will have a NaN value in the 'name' part of the table. If I used a groupby, so that I am then able to keep the dropna=False, which then allows any of my unmatched vendors to be group under the NaN part so I don't lose those units.
total_qty_by_vendor = joined.groupby('name', dropna=False)['qty'].sum().reset_index()
display(total_qty_by_vendor)

,name,qty
0,Cav Merch,1
1,Hoos Burgers,3
2,NaN,3


The groupby groups the data by the vendor's name. When I used dropna=False, Pandas doesn't get rid of the vendor name that is missing, it produces an NaN instead. Then the sum function adds up the ordered items. Lastly the reset function takes the results from Pandas and then cleans it so that flat Dataframe can be analyzed easily.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
#TODO
zone_sales = joined.groupby('zone', dropna=False)['qty'].sum().reset_index()
display(zone_sales)



,zone,qty
0,A,4
1,NaN,3


In this batch all of the vendor IDS matched, so that all seven of the units are accounted for in Zone A. If there were unmatched vendors in this they would be zoned to NaN and then the missing units would be grouped un the NaN zone, but since there was an 100% match that was not the case.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
#TODO
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]


ragged_flat_list = []
for order_info in ragged_json:
    order_id = order_info.get('order')
    vendor_id = order_info.get('vendor_id')
    lines_data = order_info.get('lines')

    if isinstance(lines_data, list) and len(lines_data) > 0:
        for line in lines_data:
            flat_row = {
                'order': order_id,
                'vendor_id': vendor_id,
                'item': line.get('item'),
                'qty': line.get('qty')
            }
            ragged_flat_list.append(flat_row)
    else:

        flat_row = {
            'order': order_id,
            'vendor_id': vendor_id,
            'item': None,
            'qty': None
        }
        ragged_flat_list.append(flat_row)

ragged_df = pd.DataFrame(ragged_flat_list)
display(ragged_df)

,order,vendor_id,item,qty
0,4,V-01,Soda,2.0
1,5,V-10,None,NaN
2,6,V-01,Fries,NaN


Here the first part of my code ensures that it loops through all of the order so that it does not crash the flattening. Then it checks to make sure hte line keys are good by doing a true or false check. Then it puts everything together into a standardized row so it is a clean Pandas DataFrame.

### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) "One row per line item" is the useful shape here, becuase it flattens the nested structure. This format makes sense because, it makes it easier to do thinks like grouping by specific item names when calculating the total quantities, analyzing the single product's popularity, and matching an indvidual item to a seperate vendors. Overall if it helpful in organizaiton and finding where patterns are when analyzing the sales. If I did this with one row per order, would force me to have intricate loop logic or list scanning in the single cell, making it more complicated than needed. This qwould result in making the relational joins and group-by aggregations really hard to do.

b) A missing quanity means there is a lack of data or there is an unknown value, like not knowing how many sodas were ordered. Whereas, a quantity of zero means exactly zero units were ordered so it provides a clear answer, rather than an unknown. If I conflate them by filling values that are not there with zeroes, then donstream mathematical calculations become wrong since I am assuming a specific number where were orginally none.